# KAT-VAD v2 — P1 / **Kill-switch K on T2** (VideoMAE V2-B)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§6.1 (Amendment 1, D7)** · Plan: `.project/plans/katvad-v2-e0-e2.md` P1.
DoTA has no pixels left, so K runs on **T2 alone**. One verdict comes out: **GO / KILL / SUSPECT_PIPELINE**.

| step | what | GPU? |
|---|---|---|
| 1 | preflight: the tree carries Amendment 1, the census and CLIP cache exist | no |
| 2 | `pick` ~300 T2-train sources (no T2-val) | no |
| 3 | sharded 7z extract → symlink farm → VideoMAE → **delete** frames | **yes** |
| 4 | `run`: probe (i′) source-grouped + (ii′) type-grouped, bootstrap over sources, verdict | no |
| 5 | record to Drive | no |

**Do not change a flag after reading §4.** The rule was committed before the first number.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_CKPT_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no checkout at {REPO} -- clone/pull branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

# core.constants reads KATVAD_* at import time -- import AFTER the block above.
from core import constants  # noqa: E402

DATASET  = constants.DADA_ORIGIN_DATASET                                  # 'DADA2000_orig'
ENCODER  = constants.VIDEOMAE_ENCODER_B                                   # pre-registered: B
STRIDE   = constants.FRAME_STRIDE                                         # 8, pairs with the CLIP cache
T2       = constants.DATA_ROOT / DATASET
CENSUS   = T2 / 'counts' / 'resolved_census.json'                         # what the T2 build used
ORIG     = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET
VIDEO_DIR = constants.VIDEO_CACHE_DIR / ENCODER / f'{DATASET}_s{STRIDE}_squash'
REPORT   = constants.OUTPUT_ROOT / 'REPORTS' / 'v2_K'
SCRATCH  = Path('/content/k')                                             # VM-local NVMe, never Drive
SCRATCH.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)
for name in ('T2', 'CENSUS', 'ORIG', 'CLIP_DIR', 'VIDEO_DIR', 'REPORT', 'SCRATCH'):
    print(f'{name:10s} {globals()[name]}')

In [ ]:
%%bash
apt-get -qq install -y p7zip-full
# transformers pinned like phase 4 (C7/C21); huggingface_hub comes with it.
pip install -q "transformers==4.56.*" av
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU'
df -h /content | tail -1

## 1. Preflight — refuse to start on a tree or data that is not what K was registered on

In [ ]:
import json
import subprocess

proc = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True)
COMMIT = proc.stdout.strip() or f'UNKNOWN (git said: {proc.stderr.strip()!r})'
branch = subprocess.run(['git', '-C', str(REPO), 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print('commit', COMMIT, '| branch', branch)

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert 'Amendment 1' in addendum and '6.1 Kill-switch K on T2' in addendum, (
    'this checkout predates Amendment 1 -- pull branch v2 before running K')
from core.tools import kill_switch_probe  # noqa: E402,F401  (fails fast if absent)
from core.data.v2_splits import load_split  # noqa: E402

val = load_split(constants.V2_SPLIT_T2_VAL)          # sha1-checked against the frozen manifest
assert (T2 / constants.META_FILENAME).is_file(), f'no T2 meta.json under {T2}'
assert CENSUS.is_file(), f'missing {CENSUS} -- phase_2 section 3 writes it'
assert (ORIG / 'DADA2000.zip').is_file(), f'no DADA2000.zip under {ORIG}'
assert (ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME).is_file()
print(f'T2-val {len(val)} sources | CLIP cache {len(list(CLIP_DIR.glob("*.npy")))} files')

## 2. Pick the K sources (no score is read)

Expect ≈ 300 picked, 0 overlap with T2-val. The list and its sha1 land on Drive **before** any feature exists.

In [ ]:
IDS = REPORT / kill_switch_probe.SOURCES_FILENAME
if IDS.exists():
    print(f'reusing {IDS} -- the pick is frozen once written; delete it only to redo the WHOLE of K')
else:
    subprocess.run([sys.executable, '-m', 'core.tools.kill_switch_probe', 'pick',
                    '--t2-meta', str(T2 / constants.META_FILENAME), '--out-dir', str(REPORT)],
                   cwd=REPO, check=True)
ids = [line.strip() for line in IDS.read_text().splitlines() if line.strip()]
pick = json.loads((REPORT / kill_switch_probe.PICK_MANIFEST_FILENAME).read_text())
assert not set(ids) & set(val), 'a T2-val source is in K'
missing_clip = [v for v in ids if not (CLIP_DIR / f'{v}.npy').exists()]
assert not missing_clip, f'{len(missing_clip)} K sources have no CLIP feature: {missing_clip[:5]}'
census = json.loads(CENSUS.read_text())
assert all(v in census for v in ids), 'a K source has no census entry'
print(f"picked {len(ids)} | sha1 {pick['sha1']} | types without a K source: {pick['types_without_k_source']}")
print(f'frames to extract ≈ {sum(census[v] for v in ids):,}')

## 3. Sharded extraction — 7z extract → farm → VideoMAE → **delete**

Resumable: a shard whose ids are all encoded is skipped. The extractor writes
`video_manifest.json` and refuses to extend a cache built with different settings (C2).
The first shard downloads the weights at the pinned commit and checks their sha256.

In [ ]:
import shutil
import time

SHARD = 60          # ~3 GiB of PNGs per shard; lower it if /content is short
BATCH = 16          # clips per forward; ViT-B at 16x224^2 fits easily on an A100


def run(cmd, cwd, capture=False):
    """Run a child and make its failure legible."""
    proc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), capture_output=capture, text=True)
    if proc.returncode:
        if capture:
            print(proc.stdout or '', proc.stderr or '', sep='\n')
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}')


def type_video(vid):
    t, v = vid.split('_')                      # t{type:02d}_v{video:03d}
    return int(t[1:]), int(v[1:])


started = time.monotonic()
for k in range(0, len(ids), SHARD):
    shard, tag = ids[k:k + SHARD], f'{k // SHARD:03d}'
    if all((VIDEO_DIR / f'{v}.npy').exists() for v in shard):
        print(f'shard {tag}: already encoded, skipping')
        continue
    frames, farm = SCRATCH / 'frames', SCRATCH / 'flat'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    farm.mkdir(parents=True)
    root = constants.DADA_ORIGIN_ROOT_DIRNAME
    images = constants.DADA_ORIGIN_IMAGES_SUBDIR
    patterns = [f'{root}/{t}/{v:03d}/{images}/*' for t, v in map(type_video, shard)]
    print(f'--- shard {tag}: {len(shard)} sources, {shutil.disk_usage("/content").free / 2**30:.1f} GiB free')
    run(['7z', 'x', 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns], cwd=ORIG)
    # C26: the bare folder name is the video number and collides across types -- link
    # each source's images dir itself under its unique id (pathlib will not recurse
    # through a symlinked parent, so the link is at the images level).
    for vid in shard:
        t, v = type_video(vid)
        src = frames / root / str(t) / f'{v:03d}' / images
        n = len(list(src.glob('*')))
        if n != census[vid]:     # C10: a folder is not evidence of data
            raise RuntimeError(f'{vid}: {n} images extracted, census says {census[vid]}')
        (farm / vid).symlink_to(src.resolve(), target_is_directory=True)
    shard_ids = SCRATCH / f'ids_{tag}.txt'
    shard_ids.write_text('\n'.join(shard) + '\n')
    run([sys.executable, '-m', 'core.tools.extract_video_features',
         '--frames-dir', farm, '--ids-file', shard_ids, '--dataset', DATASET,
         '--encoder', ENCODER, '--stride', STRIDE, '--batch-size', BATCH,
         '--device', 'auto', '--output-dir', VIDEO_DIR], cwd=REPO)
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    encoded {len(list(VIDEO_DIR.glob("*.npy")))}/{len(ids)} | '
          f'{(time.monotonic() - started) / 60:.1f} min')

In [ ]:
done = [v for v in ids if (VIDEO_DIR / f'{v}.npy').exists()]
assert len(done) == len(ids), f'{len(ids) - len(done)} K sources still unencoded -- rerun section 3'
print(json.dumps(json.loads((VIDEO_DIR / constants.VIDEO_MANIFEST_FILENAME).read_text()), indent=1))

## 4. **K** — probe, intervals, verdict (the rule is applied by the tool, not here)

Read the throughput line from §3's log into the report by hand. Expect the x-only (i′) macro in
the neighbourhood of Gate D0's 0.6518 (different subset, same label rule). **Not gated.**

In [ ]:
run([sys.executable, '-m', 'core.tools.kill_switch_probe', 'run',
     '--ids-file', IDS, '--annotation', ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME,
     '--census', CENSUS, '--clip-dir', CLIP_DIR, '--video-dir', VIDEO_DIR,
     '--out-dir', REPORT, '--commit', COMMIT], cwd=REPO)
print((REPORT / kill_switch_probe.READOUT_MD).read_text())

## 5. Record

`k_readout.{json,md}`, `k_sources.txt` and its manifest are already on Drive under
`outputs/REPORTS/v2_K/`. Copy the verdict into plan Appendix A (row P1 K) and `activeContext.md`,
with commit `COMMIT`. **Then stop**:

* **GO** → D9: the advisor picks the motion endpoint (Amendment 2) before P4.
* **KILL** → P4 is dropped; v2 = A0/A1 on DoTA s8. Tell the advisor before going on.
* **SUSPECT_PIPELINE** → debug the extractor (alignment, transform, weights); **not** a KILL.